# Advanced Forecasting Models

> **AUTHORITATIVE GENERATION — PARTIAL MODEL SET**  
> This executed notebook generates validated ARIMA rolling one-step and Prophet 30-day periodic-refit vectors. SARIMA is omitted because Step 4 found no supported weekly seasonality; PatchTST and iTransformer are deferred to an isolated Python 3.12 environment.

## 1. Protocol and Environment

ARIMA uses a true rolling one-step protocol: the model is initialized from the final 128 training returns, forecasts one day, and appends that day's observed return only after the forecast is made. Prophet uses a clearly labelled 30-day periodic-refit compromise with the latest 128 strictly prior observed prices at every refit; it is not described as rolling one-step.

The Step 4 diagnostic found lag-7 ACF `-0.0234`, within the approximate 95% bound `±0.0269`, and weekly STL seasonal strength `0.070`. Therefore no weekly seasonal component is used. A separate SARIMA result is omitted because `seasonal_order=(0,0,0,0)` would reduce to a non-seasonal ARIMA/SARIMAX specification and would not provide distinct seasonal evidence.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.bitcoin_advanced_models import (
    merge_advanced_forecasts,
    periodic_refit_prophet_forecast,
    rolling_arima_log_return_forecast,
    validate_and_save_forecast,
)
from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape
pd.set_option('display.float_format', '{:.6f}'.format)

E:\Re_Sc_AM\TimeSeriesFoundationModels\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing plotly failed. Interactive plots will not work.


## 2. Load Dataset and Freeze Split

In [2]:
data_path = PROJECT_ROOT / 'data' / 'bitcoin' / 'btcusd_1-min_data.csv'
raw = load_bitcoin_data(data_path)
daily = prepare_daily_bitcoin_data(raw)
target = daily['Close'].asfreq('D').dropna()
split_index = int(len(target) * 0.8)
train = target.iloc[:split_index]
test = target.iloc[split_index:]
assert len(train) == 4241 and len(test) == 1061
assert train.index.max() < test.index.min()
pd.DataFrame({'Segment':['Train','Test'], 'Rows':[len(train),len(test)], 'Start':[train.index.min(),test.index.min()], 'End':[train.index.max(),test.index.max()]})

,Segment,Rows,Start,End
0,Train,4241,2012-01-01 00:00:00+00:00,2023-08-11 00:00:00+00:00
1,Test,1061,2023-08-12 00:00:00+00:00,2026-07-07 00:00:00+00:00


## 3. Rolling One-Step ARIMA on Log Returns

In [3]:
arima_forecast = rolling_arima_log_return_forecast(target, test.index, context_length=128, order=(1,0,1))
arima_frame = validate_and_save_forecast(arima_forecast, test.index, PROJECT_ROOT / 'results' / 'arima_rolling_forecast.csv')
arima_metrics = pd.DataFrame([{'Model':'ARIMA Rolling One-Step','MAE':mae(test,arima_forecast),'RMSE':rmse(test,arima_forecast),'MAPE':mape(test,arima_forecast),'sMAPE':smape(test,arima_forecast)}]).set_index('Model')
arima_metrics

,MAE,RMSE,MAPE,sMAPE
Model,,,,
ARIMA Rolling One-Step,1299.874638,1866.302859,1.754004,1.754209


## 4. SARIMA Decision

No separate SARIMA model is generated. The empirical weekly-seasonality diagnostics do not support a seven-day term, and a SARIMA specification with `seasonal_order=(0,0,0,0)` would provide no seasonal mechanism and no distinct forecast vector beyond a non-seasonal ARIMA/SARIMAX model.

## 5. Prophet 30-Day Periodic Refit

In [4]:
prophet_forecast = periodic_refit_prophet_forecast(target, test.index, context_length=128, refit_every=30)
prophet_frame = validate_and_save_forecast(prophet_forecast, test.index, PROJECT_ROOT / 'results' / 'prophet_rolling_forecast.csv')
prophet_metrics = pd.DataFrame([{'Model':'Prophet 30-Day Periodic Refit','MAE':mae(test,prophet_forecast),'RMSE':rmse(test,prophet_forecast),'MAPE':mape(test,prophet_forecast),'sMAPE':smape(test,prophet_forecast)}]).set_index('Model')
prophet_metrics

15:23:49 - cmdstanpy - INFO - Chain [1] start processing


15:23:50 - cmdstanpy - INFO - Chain [1] done processing


15:23:51 - cmdstanpy - INFO - Chain [1] start processing


15:23:51 - cmdstanpy - INFO - Chain [1] done processing


15:23:52 - cmdstanpy - INFO - Chain [1] start processing


...

15:24:27 - cmdstanpy - INFO - Chain [1] start processing


15:24:27 - cmdstanpy - INFO - Chain [1] done processing


15:24:28 - cmdstanpy - INFO - Chain [1] start processing


15:24:28 - cmdstanpy - INFO - Chain [1] done processing


,MAE,RMSE,MAPE,sMAPE
Model,,,,
Prophet 30-Day Periodic Refit,8195.262862,10781.162873,11.199767,11.287185


## 6. Deferred NeuralForecast Models

PatchTST and iTransformer via neuralforecast were not run in this environment. neuralforecast>=3.1.8 requires pytorch-lightning<2.6.0 (incompatible with the project's pinned lightning 2.6.5); neuralforecast==3.1.7 is compatible with lightning but imports ray[train,tune], which has no available Windows/Python 3.13 distribution in this environment as of this run. A separate Python 3.12 environment (the same fix already planned for Moirai/uni2ts) is the recommended path if these models are needed later.

## 7. Merge and Validate Artifacts

In [5]:
validated_path = PROJECT_ROOT / 'results' / 'validated_forecasts.csv'
validated = merge_advanced_forecasts(validated_path, [arima_frame, prophet_frame])
checks = pd.DataFrame({
    'Check':['ARIMA rows','Prophet rows','ARIMA aligned','Prophet aligned','No NaNs','Finite values','No duplicate vectors'],
    'Pass':[len(arima_frame)==1061,len(prophet_frame)==1061,pd.DatetimeIndex(arima_frame.Timestamp).equals(test.index),pd.DatetimeIndex(prophet_frame.Timestamp).equals(test.index),not validated.isna().any().any(),np.isfinite(validated.select_dtypes(include=[np.number])).all().all(),not np.array_equal(validated['ARIMA_Rolling'].to_numpy(),validated['Prophet_Periodic_Refit'].to_numpy())]
})
assert checks['Pass'].all()
checks

,Check,Pass
0,ARIMA rows,True
1,Prophet rows,True
2,ARIMA aligned,True
3,Prophet aligned,True
4,No NaNs,True
5,Finite values,True
6,No duplicate vectors,True


## 8. Final Numeric Comparison

In [6]:
comparison = pd.concat([arima_metrics, prophet_metrics]).sort_values('RMSE')
comparison

,MAE,RMSE,MAPE,sMAPE
Model,,,,
ARIMA Rolling One-Step,1299.874638,1866.302859,1.754004,1.754209
Prophet 30-Day Periodic Refit,8195.262862,10781.162873,11.199767,11.287185
